<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/sae/SAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sae-lens transformer-lens circuitsvis

### Load the model

In [2]:
import numpy as np
import pandas as pd

In [3]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [4]:
from huggingface_hub import hf_hub_download

REPO_ID = "sojup/entity_binding_test"
FILENAME = "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
# weights_path = "/content/D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

In [5]:
import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example(num_relations: int, *, allow_self_loops: bool = False):
    facts = []
    seen_head_rel = set()
    seen_e = set()
    seen_t = set()

    while len(facts) < num_relations:
        e = int(rng.integers(0, E))
        t = int(TYPES[rng.integers(0, T)])

        # enforce uniqueness of e and t (not just the tuple)
        if e in seen_e or t in seen_t:
            continue
        if (e, t) in seen_head_rel:
            continue

        # forbid self-loop (optional)
        e2 = int(rng.integers(0, E))
        while not allow_self_loops and e2 == e:
            e2 = int(rng.integers(0, E))

        seen_head_rel.add((e, t))
        seen_e.add(e)
        seen_t.add(t)
        facts.append((e, t, e2))

    q_idx = int(rng.integers(0, num_relations))
    Eq, Tq, E2q = facts[q_idx]

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])
    seq.extend([Tq, Eq, Q])

    return seq, E2q, facts



rows = []
for _ in tqdm(range(N_WORLDS)):
    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label, _ = produce_example(k, allow_self_loops=False)
    rows.append({"tokens": seq, "label": label})

df = pd.DataFrame(rows)

100%|██████████| 80000/80000 [00:08<00:00, 9623.16it/s] 


In [6]:
N_LAYERS = 3
HEADS = 2

d_model = 256
d_mlp   = 1024
n_ctx   = 64
lr = 3e-4
betas = (0.9, 0.98)
weight_decay = 0.01
num_epochs = 30

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        d_mlp=d_mlp,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

Moving model to device:  cpu
Model loaded successfully.


In [15]:
D_VOCAB

113

In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [8]:
from sklearn.model_selection import train_test_split
from datasets.arrow_dataset import Dataset # Moved import here to avoid circular dependency

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

# test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

### Train a SAE

In [9]:
from sae_lens import (
    LanguageModelSAERunnerConfig,
    SAETrainingRunner,
    StandardTrainingSAEConfig,
    LoggingConfig,
)

In [10]:
import os
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Using device: cpu


In [11]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [30]:
model is None

False

In [35]:
import traceback
total_training_steps = 1000 # we should do more
batch_size = 1
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = 0
lr_decay_steps = total_training_steps // 5  # 20% of training
l1_warm_up_steps = total_training_steps // 20  # 5% of training

cfg = LanguageModelSAERunnerConfig(
    # Data Generating Function (Model + Training Distibuion)
    model_name=None,  # our model (more options here: https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html)
    model_class_name="HookedTransformer",
    # model_name="sojup/entity_binding_test/D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt",  # our model (more options here: https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html)
    hook_name="blocks.2.hook_resid_post",  # A valid hook point (see more details here: https://neelnanda-io.github.io/TransformerLens/generated/demos/Main_Demo.html#Hook-Points)
    dataset_path="sojup/entity_binding",  # this is a tokenized language dataset on Huggingface for the Tiny Stories corpus.
    is_dataset_tokenized=True,
    streaming=True,  # we could pre-download the token dataset if it was small.
    # SAE Parameters
    sae=StandardTrainingSAEConfig(
        d_in=256,  # the width of the mlp output.
        d_sae=16384,  # the width of the SAE. Larger will result in better stats but slower training.
        apply_b_dec_to_input=False,  # We won't apply the decoder weights to the input.
        normalize_activations="expected_average_only_in",
        l1_coefficient=5,  # will control how sparse the feature activations are
        l1_warm_up_steps=l1_warm_up_steps,  # this can help avoid too many dead features initially.
    ),
    # Training Parameters
    lr=5e-5,  # lower the better, we'll go fairly high to speed up the tutorial.
    adam_beta1=0.9,  # adam params (default, but once upon a time we experimented with these.)
    adam_beta2=0.999,
    lr_scheduler_name="constant",  # constant learning rate with warmup. Could be better schedules out there.
    lr_warm_up_steps=lr_warm_up_steps,  # this can help avoid too many dead features initially.
    lr_decay_steps=lr_decay_steps,  # this will help us avoid overfitting.
    train_batch_size_tokens=batch_size,
    context_size=19,
    #512,  # will control the lenght of the prompts we feed to the model. Larger is better but slower. so for the tutorial we'll use a short one.
    # Activation Store Parameters
    n_batches_in_buffer=64,  # controls how many activations we store / shuffle.
    training_tokens=total_training_tokens,  # 100 million tokens is quite a few, but we want to see good stats. Get a coffee, come back.
    store_batch_size_prompts=16,
    # Resampling protocol
    feature_sampling_window=1000,  # this controls our reporting of feature sparsity stats
    dead_feature_window=1000,  # would effect resampling or ghost grads if we were using it.
    dead_feature_threshold=1e-4,  # would effect resampling or ghost grads if we were using it.
    # WANDB
    # logger=LoggingConfig(
    #     log_to_wandb=True,  # always use wandb unless you are just testing code.
    #     wandb_project="sae_lens_tutorial",
    #     wandb_log_frequency=30,
    #     eval_every_n_wandb_logs=20,
    # ),
    # Misc
    device=device,
    seed=42,
    n_checkpoints=0,
    checkpoint_path="checkpoints",
    dtype="float32",
    use_cached_activations = False,
    compile_llm=False,
    # n_eval_batches=0
)
# look at the next cell to see some instruction for what to do while this is running.
sparse_autoencoder = SAETrainingRunner(cfg, override_model=model)
try:
  sparse_autoencoder.run()
except:
  traceback.print_exc()

/tmp/ipython-input-1175220431.py:64: DeprecationWarning: Use LanguageModelSAETrainingRunner instead
  sparse_autoencoder = SAETrainingRunner(cfg, override_model=model)
/usr/local/lib/python3.11/dist-packages/sae_lens/saes/sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


details/current_learning_rate,███████████████████████████████████▅▅▃▂▁
details/l1_coefficient,▁▆██████████████████████████████████████
details/n_training_samples,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
losses/l1_loss,█▆▇▃▂▁▄▁▁▃▄▄▃▄▅▄▂▂▂▄▁▂▃▁▂▂▃▂▂▄▂▃▂▅▂▂▄▃▂▂
losses/mse_loss,▂▄▂▄▄▄▃▂▅▃▁▁▂▂▅▂▃▃▃▂▂▂▃█▂▂▃▁▃▄█▃▁▁▂▂▄▅▂▄
losses/overall_loss,▇▅█▇▅▄▂▃▆▅▂▄▅▇▃▃▅▄▂▅▄▃▂▃▂▄▁▅▁█▄▄▄▂▄▆▁▁▃▂
metrics/l0,██▂▂▃▁▁▁▂▁▂▁▁▁▁▁▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/mean_log10_feature_sparsity,▁
sparsity/below_1e-5,▁
sparsity/below_1e-6,▁
sparsity/dead_features,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁


Training SAE:   0%|          | 0/1000 [00:00<?, ?it/s]

Estimating norm scaling factor:   0%|          | 0/1000 [00:00<?, ?it/s]











Refilling buffer:   0%|          | 0/32 [00:00<?, ?it/s]









Refilling buffer:   3%|▎         | 1/32 [00:00<00:06,  5.10it/s]









Refilling buffer:  16%|█▌        | 5/32 [00:00<00:01, 19.25it/s]









Refilling buffer:  31%|███▏      | 10/32 [00:00<00:00, 29.23it/s]









Refilling buffer:  47%|████▋     | 15/32 [00:00<00:00, 34.18it/s]









Refilling buffer:  59%|█████▉    | 19/32 [00:00<00:00, 34.87it/s]









Refilling buffer:  75%|███████▌  | 24/32 [00:00<00:00, 37.86it/s]









Refilling buffer:  91%|█████████ | 29/32 [00:00<00:00, 39.70it/s]









                                                                 /usr/local/lib/python3.11/dist-packages/sae_lens/training/sae_trainer.py:330: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  "metrics/explained_variance_lega

In [36]:
%debug
#[16, 18, 100]

> /usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py(1219)_input_request()
   1217             except KeyboardInterrupt:
   1218                 # re-raise KeyboardInterrupt, to truncate traceback
-> 1219                 raise KeyboardInterrupt("Interrupted by user") from None
   1220             except Exception:
   1221                 self.log.warning("Invalid Message:", exc_info=True)

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user


In [13]:
# from datasets import Dataset, DatasetDict

# # Create Dataset objects from pandas DataFrames
# train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
# val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
# test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# # Create a DatasetDict
# dataset_dict = DatasetDict({
#     'train': train_dataset,
#     'validation': val_dataset,
#     'test': test_dataset
# })

# print(dataset_dict)
  # from huggingface_hub import notebook_login
  # notebook_login()

  # repo_name = "sojup/entity_binding"

  # # You will need to be logged in to Hugging Face to push to the hub


  # # Push the dataset to the Hugging Face Hub
  # dataset_dict.push_to_hub(repo_name)

  # print(f"DatasetDict created. Uncomment the notebook_login() and push_to_hub() lines and replace '{repo_name}' with your desired repository name to upload to the Hugging Face Hub.")